# Sesten duygu: omurga (backbone) seçimi — Aşama 1

Bu not defteri aday modelleri **aynı Türkçe veride, aynı koşulda** yarıştırır. İnce ayar (Aşama 2) hangi omurgayla yapılacaksa
ona burada karar verilir.

* **Veri:** TurEV-DB (6 konuşmacı, 1735 kayıt, 4 duygu). Lisans CC BY-NC-ND: yalnızca araştırma/PoC.
* **Yöntem:** model donmuş; her katmandan kayıt başına ortalama gömme → lojistik regresyon. **Konuşmacı-dışarıda** doğrulama:
  her seferinde bir konuşmacı test, katman seçimi yalnızca eğitim konuşmacılarıyla yapılır.
* **Ölçütler:** 4 sınıf UAR, uyarılma ve valens ikili doğruluk (Russell'ın çember modeli), CPU hızı (RTF).

**Çalıştırma ortamı:** Çalışma zamanı → Çalışma zamanı türünü değiştir → **T4 GPU**. Tüm modeller ~15-25 dakika.

In [ ]:
# 1) Kod ve veri
import os
REPO = "https://github.com/Pappates/Emotional_Feedback_Costumer_Service"
# Depo özelse: from getpass import getpass; REPO = REPO.replace("https://", f"https://{getpass('GitHub token: ')}@")
if not os.path.exists("proje"):
    !git clone -q {REPO} proje
if not os.path.exists("turev"):
    !git clone -q --depth 1 https://github.com/Xeonen/TurEV-DB turev
!pip -q install opensmile funasr librosa soundfile scikit-learn
import torch; print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "YOK (T4 seçin)")

## 2) Öznitelik çıkarımı

| Aday | Neden aday | Boyut |
|---|---|---|
| `egemaps` | Psikoakustik standart özellik seti (Eyben 2016), açıklanabilir taban çizgisi | 88 özellik |
| `superb_er` | Şu an uygulamada kullanılan ses modeli (İngilizce, 4 duygu) | 95M |
| `whisper_base`, `whisper_small` | Türkçeyi bilen konuşma tanıma modeli; transkripsiyon için zaten çalışıyor → duygu için ek maliyet ~0 | 74M / 244M |
| `emotion2vec_plus_base/large` | Duyguya özel ön eğitimli evrensel temsil | 90M / 300M |
| `mhubert_147` | 147 dilde ön eğitimli HuBERT (Türkçe dahil) | 95M |
| `wavlm_base_plus` | Konuşma işleme kıyaslamalarında güçlü (İngilizce ön eğitim) | 95M |
| `xlsr_300m` | 128 dilde ön eğitimli wav2vec2 | 300M |
| `audeering_dim` | MSP-Podcast'te valens/uyarılma/**baskınlık** için ince ayarlı (CC BY-NC-SA) | 300M |

Whisper burada üretimdeki gibi 30 sn dolguyla çalışır (CPU'da çok yavaş olduğu için bu not defterinde ölçülür).

In [ ]:
MODELLER = "egemaps superb_er whisper_base whisper_small emotion2vec_plus_base emotion2vec_plus_large mhubert_147 wavlm_base_plus xlsr_300m audeering_dim"
!cd proje/egitim && HF_HUB_DISABLE_XET=1 python ser_bench.py "/content/turev/Sound Source" /content/ser_cache {MODELLER} 2>&1 | grep -v "Warning\|Loading" 

## 3) Değerlendirme (konuşmacı-dışarıda)

In [ ]:
!cd proje/egitim && python ser_eval.py /content/ser_cache

## 4) Nasıl okunur?

* **UAR(4)**: dört duyguda dengeli doğruluk. Şans düzeyi 0,25.
* **±konuşmacı**: konuşmacılar arası sapma. Düşük olması, modelin yeni seslere genellendiğini gösterir; çağrı merkezinde her arayan yenidir.
* **uyarılma / valens**: psikolojik boyutlar. Literatürde ses uyarılmayı iyi, valensi zayıf taşır; valens için metinle füzyon gerekir.
* **RTF** burada GPU'da ölçülür; CPU hızı için yerel ölçümlere bakın (README).
* **katman**: en iyi bilgi genelde orta-üst katmanlardadır. İnce ayarda bu katmanlara ağırlık verilir (ağırlıklı katman toplamı).

Sonuç dosyası: `/content/ser_cache/sonuc.json` (Drive'a kopyalayabilirsiniz).

In [ ]:
from google.colab import files; files.download("/content/ser_cache/sonuc.json")